# Temporal Alignment — News and Liquidity Data

Loads all articles from the `articles` table, assigns each to its next available
trading hour in `market_context` using `merge_asof(direction='forward')`, left-joins
`llm_features`, and writes the result to the `liquidity` table.

**Key design decisions:**
- `dt.ceil('h')` preserves causality — never assigns article to hour before publication
- `merge_asof(direction='forward')` vectorized O(n log n) replacement for the old `.apply()` loop
- Inner join to `market_context`: articles outside coverage are dropped and counted
- Left join `llm_features`: articles without LLM scores are kept, flagged with `has_llm_features=0`
- `assignment_gap` records hours between publication timestamp and assigned trading hour
- All reads and writes use `wti_thesis.db` directly — no CSV reads or writes

### If notebook 06 is re-run after this notebook

Run order: this notebook depends on llm_features being populated by notebook 06. Re-run this notebook after every batch of LLM extractions to refresh the liquidity table.

### General imports

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import datetime as dt

### Load all data from DB

In [8]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

df_articles = pd.read_sql("""
    SELECT id, title, datetime, domain, url
    FROM articles
""", conn)

df_mc = pd.read_sql("""
    SELECT datetime_hour, log_volume, price_range, log_return, amihud, dxy, vix
    FROM market_context
    WHERE log_volume > 0
""", conn)


df_llm = pd.read_sql("""
    SELECT article_id, usable, sentiment_score, supply_impact, demand_impact,
           risk_premium, magnitude, event_type, entities, certainty, time_horizon
    FROM llm_features
""", conn)

conn.close()

df_articles['datetime'] = pd.to_datetime(df_articles['datetime'], utc=True)
df_mc['datetime_hour'] = pd.to_datetime(df_mc['datetime_hour'], utc=True)
df_mc = df_mc.sort_values('datetime_hour').reset_index(drop=True)

print(f"Articles:                        {len(df_articles):,}")
print(f"Market context hours (vol > 0):  {len(df_mc):,}")
print(f"LLM features:                    {len(df_llm):,}")
print(f"Market context range: {df_mc['datetime_hour'].min()} → {df_mc['datetime_hour'].max()}")

Articles:                        22,795
Market context hours (vol > 0):  10,834
LLM features:                    19,619
Market context range: 2024-05-13 12:00:00+00:00 → 2026-05-13 11:00:00+00:00


### Assign each article to its next available trading hour

`dt.ceil('h')` gives the earliest possible honest hour — the article cannot have
influenced traders before it was published. `merge_asof(direction='forward')` then
finds the smallest `datetime_hour >= datetime_ceil`, which forward-assigns
off-hours articles (nights, weekends) to the next market open instead of discarding them.

In [9]:
df_articles = df_articles.sort_values('datetime').reset_index(drop=True)
df_articles['datetime_ceil'] = df_articles['datetime'].dt.ceil('h')

# Forward-assign: smallest datetime_hour >= datetime_ceil (inner join semantics via dropna)
df_aligned = pd.merge_asof(
    df_articles,
    df_mc[['datetime_hour', 'log_volume', 'price_range', 'log_return', 'amihud', 'dxy', 'vix']],
    left_on='datetime_ceil',
    right_on='datetime_hour',
    direction='forward'
)

total = len(df_aligned)
df_aligned = df_aligned.dropna(subset=['datetime_hour']).reset_index(drop=True)
dropped = total - len(df_aligned)

df_aligned['assignment_gap'] = (
    df_aligned['datetime_hour'] - df_aligned['datetime']
).dt.total_seconds() / 3600

contemp = (df_aligned['assignment_gap'] < 2).sum()
fwd = (df_aligned['assignment_gap'] >= 2).sum()

print(f"Total articles:          {total:,}")
print(f"Aligned (in range):      {len(df_aligned):,}")
print(f"Dropped (out of range):  {dropped:,}")
print(f"Contemporaneous (<2h):   {contemp:,}")
print(f"Forward-assigned (>=2h): {fwd:,}")
print(f"Date range: {df_aligned['datetime_hour'].min()} → {df_aligned['datetime_hour'].max()}")

Total articles:          22,795
Aligned (in range):      22,795
Dropped (out of range):  0
Contemporaneous (<2h):   15,290
Forward-assigned (>=2h): 7,505
Date range: 2024-05-13 12:00:00+00:00 → 2026-05-12 01:00:00+00:00


### Sanity checks

In [10]:
BASELINE_ALIGNED = 22000

assert len(df_aligned) >= BASELINE_ALIGNED, (
    f"STOP: {len(df_aligned):,} aligned rows — expected >= {BASELINE_ALIGNED:,}"
)
assert len(df_aligned) <= total, (
    f"STOP: {len(df_aligned):,} aligned > {total:,} total — merge error"
)
assert df_aligned['log_volume'].isna().sum() == 0, (
    f"STOP: {df_aligned['log_volume'].isna().sum()} rows with NULL log_volume"
)
print(f"Sanity checks passed — {len(df_aligned):,} aligned rows")

Sanity checks passed — 22,795 aligned rows


### Left-join LLM features

In [11]:
df_aligned = df_aligned.merge(
    df_llm,
    left_on='id',
    right_on='article_id',
    how='left'
)
df_aligned['has_llm_features'] = df_aligned['usable'].notna().astype(int)
df_aligned['usable'] = df_aligned['usable'].fillna(0).astype(int)

with_llm = df_aligned['has_llm_features'].sum()
without_llm = len(df_aligned) - with_llm
print(f"With LLM features:    {with_llm:,}")
print(f"Without LLM features: {without_llm:,}")

With LLM features:    19,619
Without LLM features: 3,176


### Write liquidity table to DB

In [12]:
df_liquidity = df_aligned[[
    'id', 'datetime_hour', 'assignment_gap',
    'log_volume', 'price_range', 'log_return', 'amihud', 'dxy', 'vix',
    'usable', 'sentiment_score', 'supply_impact', 'demand_impact', 'risk_premium',
    'magnitude', 'event_type', 'entities',
    'certainty', 'time_horizon', 'has_llm_features'
]].copy().rename(columns={'id': 'article_id'})


df_liquidity['datetime_hour'] = df_liquidity['datetime_hour'].astype(str)

conn = sqlite3.connect("../01_data/wti_thesis.db")
conn.execute("DROP TABLE IF EXISTS liquidity")
conn.commit()
df_liquidity.to_sql('liquidity', conn, if_exists='replace', index=False)
count = conn.execute("SELECT COUNT(*) FROM liquidity").fetchone()[0]
conn.close()

print(f"Liquidity table written: {count:,} rows")

Liquidity table written: 22,795 rows


### Verify

In [13]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

stats = pd.read_sql("""
    SELECT
        COUNT(*)                                              AS total_aligned,
        SUM(CASE WHEN assignment_gap < 2  THEN 1 ELSE 0 END) AS contemporaneous,
        SUM(CASE WHEN assignment_gap >= 2 THEN 1 ELSE 0 END) AS forward_assigned,
        SUM(has_llm_features)                                 AS with_llm,
        SUM(1 - has_llm_features)                             AS without_llm,
        MIN(datetime_hour)                                    AS date_min,
        MAX(datetime_hour)                                    AS date_max
    FROM liquidity
""", conn)
print(stats.to_string())

sample = pd.read_sql("""
    SELECT a.title, a.datetime, l.datetime_hour, l.log_volume, l.assignment_gap
    FROM articles a
    JOIN liquidity l ON a.id = l.article_id
    ORDER BY a.datetime DESC
    LIMIT 5
""", conn)
print(sample.to_string())

conn.close()
print("\nAlignment complete.")

   total_aligned  contemporaneous  forward_assigned  with_llm  without_llm                   date_min                   date_max
0          22795            15290              7505     19619         3176  2024-05-13 12:00:00+00:00  2026-05-12 01:00:00+00:00
                                                                                                            title                   datetime              datetime_hour  log_volume  assignment_gap
0                                             Letters : Supporting social workers will help keep foster kids safe  2026-05-12 00:30:00+00:00  2026-05-12 01:00:00+00:00    7.760041            0.50
1                                                 Saudi Aramco CEO warns oil markets may not normalize until 2027  2026-05-12 00:30:00+00:00  2026-05-12 01:00:00+00:00    7.760041            0.50
2                                                                              Remittances expose Gulf dependency  2026-05-11 23:30:00+00:00  2026-05-12 0

In [14]:
conn = sqlite3.connect("../01_data/wti_thesis.db")
v = conn.execute("""
    SELECT
        COUNT(*)                                                          AS total_rows,
        SUM(CASE WHEN usable=1                        THEN 1 ELSE 0 END) AS usable_true,
        SUM(CASE WHEN has_llm_features=1 AND usable=0 THEN 1 ELSE 0 END) AS llm_rejected,
        SUM(CASE WHEN has_llm_features=0              THEN 1 ELSE 0 END) AS no_body,
        SUM(CASE WHEN supply_impact IS NOT NULL        THEN 1 ELSE 0 END) AS with_supply,
        SUM(CASE WHEN demand_impact IS NOT NULL        THEN 1 ELSE 0 END) AS with_demand,
        SUM(CASE WHEN risk_premium  IS NOT NULL        THEN 1 ELSE 0 END) AS with_risk,
        SUM(has_llm_features)                                             AS with_llm
    FROM liquidity
""").fetchone()
conn.close()

total_rows, usable_true, llm_rejected, no_body, with_supply, with_demand, with_risk, with_llm = v

print(f"=== Phase 5 liquidity verification ===")
print(f"Total rows:                         {total_rows:,}")
print(f"  modeling-ready  (usable=1):       {usable_true:,}  ← canonical downstream filter")
print(f"  LLM-rejected    (usable=0):       {llm_rejected:,}")
print(f"  no-body         (no LLM called):  {no_body:,}")
print()
print(f"Channel coverage (must each equal usable=1 = {usable_true:,}):")
print(f"  supply_impact non-null:           {with_supply:,}")
print(f"  demand_impact non-null:           {with_demand:,}")
print(f"  risk_premium  non-null:           {with_risk:,}")

assert with_supply == with_demand == with_risk == usable_true, (
    f"Channel coverage mismatch: supply={with_supply}, demand={with_demand}, "
    f"risk={with_risk}, usable=1={usable_true}"
)
assert with_llm == usable_true + llm_rejected, (
    f"has_llm_features={with_llm} != usable_true({usable_true}) + llm_rejected({llm_rejected})"
)
assert total_rows == usable_true + llm_rejected + no_body, (
    f"Row count mismatch: {total_rows} != {usable_true} + {llm_rejected} + {no_body}"
)
print("\nAll assertions passed.")

=== Phase 5 liquidity verification ===
Total rows:                         22,795
  modeling-ready  (usable=1):       11,675  ← canonical downstream filter
  LLM-rejected    (usable=0):       7,944
  no-body         (no LLM called):  3,176

Channel coverage (must each equal usable=1 = 11,675):
  supply_impact non-null:           11,675
  demand_impact non-null:           11,675
  risk_premium  non-null:           11,675

All assertions passed.


### Update project logbook

In [15]:
entry = f"""
---

## Phase 5 — Notebook 05 alignment run ({dt.date.today()})

### Alignment results

**Schema:**
- `liquidity` includes: `usable`, `supply_impact`, `demand_impact`, `risk_premium`, `dxy`, `vix`
- `price_direction` dropped; `body_valid` no longer propagated (superseded by `usable`)
- `usable=1` is the canonical downstream filter for modeling

**Pipeline:**
- All reads from `wti_thesis.db` — no CSV reads
- Vectorized `merge_asof(direction='forward')` replaces the old `.apply()` loop
- Left join `llm_features` → `has_llm_features` and `usable` flags
- Sanity check: assert aligned >= 22,000 before writing
- `DROP TABLE IF EXISTS liquidity` before write — no stale rows

**Results:**
- Total articles in DB: {total:,}
- Aligned to market_context: {len(df_aligned):,}
- Dropped (out of range): {dropped:,}
- Contemporaneous (<2h gap): {contemp:,}
- Forward-assigned (>=2h): {fwd:,}
- Modeling-ready (usable=1): {usable_true:,}  ← canonical filter
- LLM-rejected (usable=0, LLM called): {llm_rejected:,}
- No-body (no LLM called): {no_body:,}

**Channel coverage:**
- supply_impact, demand_impact, risk_premium non-null iff usable=1 ({usable_true:,} rows)
- All three channel counts verified equal to usable=1 count
"""

logbook_path = "../05_reports/development-decisions/project_logbook.md"
with open(logbook_path, 'a') as f:
    f.write(entry)

print(f"Logbook updated: {logbook_path}")

# ── TFT v2 Prep Task 1 — one-time record, written on the run that added dxy/vix ──
task1_note = """
---

## TFT v2 Prep — Task 1: DXY/VIX propagated to liquidity (2026-05-30)

**Change:** `dxy` and `vix` were missing from `liquidity` because the Phase 5 DB migration
selected only `log_volume, price_range, log_return, amihud` from `market_context`.
Fixed by adding both columns to:
1. `market_context` SELECT in the load cell
2. `merge_asof` column projection in the assign cell
3. `df_liquidity` column list in the write cell

**Verification (2026-05-30):**
```sql
SELECT COUNT(*),
       SUM(CASE WHEN dxy IS NOT NULL THEN 1 ELSE 0 END) AS with_dxy,
       SUM(CASE WHEN vix IS NOT NULL THEN 1 ELSE 0 END) AS with_vix
FROM liquidity;
```
- total: 22,795 | with_dxy: 22,794 | with_vix: 22,788 ✅
- Gap of 1 / 7 rows matches the ~0.1% off-hours coverage holes in market_context.

**Next:** Task 2 — entity normalization pass.
"""

# Guard: only append if this note isn't already in the logbook
with open(logbook_path) as f:
    existing = f.read()

if "TFT v2 Prep — Task 1" not in existing:
    with open(logbook_path, 'a') as f:
        f.write(task1_note)
    print("Task 1 note appended.")
else:
    print("Task 1 note already present — skipped.")

Logbook updated: ../05_reports/development-decisions/project_logbook.md
Task 1 note appended.
